In [3]:
import os
from dotenv import load_dotenv
import requests
import re
from datasets import load_dataset
from user_api import *
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

API_KEY = os.environ.get("API_KEY")

In [4]:
steamid = extraer_steamid("gaben")
print(steamid)

2026-03-18 09:49:44,411 - INFO - Alias directo detectado: gaben
2026-03-18 09:49:44,412 - INFO - Resolviendo alias: gaben
2026-03-18 09:49:44,653 - INFO - Alias resuelto: 76561197968052866


76561197968052866


In [5]:
obtener_juegos_usuario(steamid)

2026-03-18 09:49:44,660 - INFO - Consultando juegos del usuario 76561197968052866
2026-03-18 09:49:44,660 - INFO - Enviando petición a la API de Steam
2026-03-18 09:49:44,894 - INFO - Código de respuesta: 200
2026-03-18 09:49:44,894 - INFO - Se encontraron 0 juegos


""


In [6]:
# -------------------------------------------------
# 4️⃣ Ejemplo de uso
# -------------------------------------------------

url_usuario = input("https://steamcommunity.com/profiles/76561198367022349/")

steamid = extraer_steamid(url_usuario)

if not steamid:
    print("❌ No se pudo obtener el SteamID.")
else:
    print("✅ SteamID detectado:", steamid)
    
    juegos_usuario = obtener_juegos_usuario(steamid)
    
    df_juegos_usuario = pd.DataFrame(juegos_usuario)

    if not juegos_usuario:
        print("⚠️ No se encontraron juegos (perfil privado o sin juegos).")
    else:
        print(f"🎮 Juegos encontrados: {len(juegos_usuario)}\n")
        
        # Mostrar primeros 10
        for juego in juegos_usuario[:10]:
            print(f"- {juego['name']} ({juego['playtime_forever']} minutos)")

2026-03-18 09:49:50,508 - INFO - SteamID numérico detectado
2026-03-18 09:49:50,508 - INFO - Consultando juegos del usuario 76561198367022349
2026-03-18 09:49:50,508 - INFO - Enviando petición a la API de Steam


✅ SteamID detectado: 76561198367022349


2026-03-18 09:49:51,252 - INFO - Código de respuesta: 200
2026-03-18 09:49:51,253 - INFO - Se encontraron 76 juegos


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
try:
    df = pd.read_parquet("data/raw/games.parquet")
    print("Cargado desde Parquet.")
except Exception:
    df = pd.read_pickle("data/raw/games.pkl")
    print("Cargado desde Pickle.")

Cargado desde Parquet.


In [ ]:
perfil_usuario = construir_perfil_usuario(df, juegos_usuario)

2026-03-15 00:34:47,617 - INFO - Juegos con al menos 120 minutos jugados: 51
2026-03-15 00:34:47,618 - INFO - AppIDs válidos: 51
2026-03-15 00:34:47,638 - INFO - Juegos encontrados en dataset: 43


In [ ]:
df['app_id'].dtype

<StringDtype(na_value=nan)>

In [ ]:
df_juegos_usuario['appid'].dtype

dtype('int64')

In [ ]:
# Mostrar algunos juegos encontrados
print("\n🔎 Algunos juegos detectados en tu dataset:")
print(perfil_usuario[["name", "genres", "positive", "negative"]].head())

# Calcular rating real
perfil_usuario["rating_real"] = (
    perfil_usuario["positive"] /
    (perfil_usuario["positive"] + perfil_usuario["negative"])
)

print("\n⭐ Juegos mejor valorados del usuario:")
print(
    perfil_usuario
    .sort_values("rating_real", ascending=False)
    [["name", "rating_real"]]
    .head()
)


🔎 Algunos juegos detectados en tu dataset:
                      name                                             genres  \
1581           Idle Slayer            [Casual, Indie, Strategy, Free To Play]   
3692              PAYDAY 2                                      [Action, RPG]   
4029          Conan Exiles  [Action, Adventure, Massively Multiplayer, RPG...   
8878   PUBG: BATTLEGROUNDS  [Action, Adventure, Massively Multiplayer, Fre...   
17066       Rocket League®                    [Action, Indie, Racing, Sports]   

       positive  negative  
1581       7488      1279  
3692     594327     68755  
4029      84575     22890  
8878    1520457   1037487  
17066    509445     73775  

⭐ Juegos mejor valorados del usuario:
                              name  rating_real
81197               Slay the Spire     0.977907
25026   Doki Doki Literature Club!     0.964550
39800     The Witcher 3: Wild Hunt     0.961266
109527                  God of War     0.960061
45588                 

In [9]:
print("\n⏳ Juegos más jugados del usuario:")
print(
    df_juegos_usuario
    .sort_values("playtime_forever", ascending=False)
    [["name", "playtime_forever"]]
    .head()
)


⏳ Juegos más jugados del usuario:
                   name  playtime_forever
68             Lost Ark            584217
35                SMITE            144652
33     Dead by Daylight             87426
66          Idle Slayer             42867
44  PUBG: BATTLEGROUNDS             42733


In [ ]:
from collections import Counter

# aplanar listas de géneros
generos = [gen for sublist in perfil_usuario["genres"] for gen in sublist]

# contar géneros
contador_generos = Counter(generos)

print("🎯 Géneros favoritos del usuario:")
for gen, count in contador_generos.most_common(10):
    print(f"{gen}: {count}")

🎯 Géneros favoritos:
Action: 26
Strategy: 19
Adventure: 18
RPG: 17
Free To Play: 15
Indie: 12
Simulation: 9
Casual: 6
Massively Multiplayer: 6
Early Access: 3


In [11]:
print("Número de juegos en el perfil del usuario:", len(perfil_usuario))
print("Primeros 5 juegos:")
print(perfil_usuario[["name", "genres"]].head())

Número de juegos en el perfil del usuario: 43
Primeros 5 juegos:
                      name                                             genres
1581           Idle Slayer            [Casual, Indie, Strategy, Free To Play]
3692              PAYDAY 2                                      [Action, RPG]
4029          Conan Exiles  [Action, Adventure, Massively Multiplayer, RPG...
8878   PUBG: BATTLEGROUNDS  [Action, Adventure, Massively Multiplayer, Fre...
17066       Rocket League®                    [Action, Indie, Racing, Sports]


In [20]:
from collections import Counter
import re

all_tags = []

for t in perfil_usuario["tags"]:
    if not t:
        continue
    # convertir a string
    t_str = str(t)
    # buscar solo la parte del tag antes de ":"
    tags_list = [x.split(":")[0].strip() for x in t_str.split(",") if x.strip()]
    all_tags.extend(tags_list)

contador_tags = Counter(all_tags)

print("\n🏷️ Tags favoritos:")
for tag, count in contador_tags.most_common(15):
    print(f"{tag}: {count}")


🏷️ Tags favoritos:
'Singleplayer: 31
'Action: 31
'Multiplayer: 29
'Co-op: 20
'Strategy: 19
'Adventure: 18
'RPG: 16
'Atmospheric: 16
'Online Co-Op: 15
'Great Soundtrack: 15
'Tactical: 13
'PvP: 13
'Fantasy: 12
'Third Person: 12
'Indie: 11


In [34]:
# 1️⃣ Construir contenido (sin tags)
df["content"] = (
    df["genres"].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x)) + " " +
    df["short_description"].fillna("")
)

# 2️⃣ Vectorizar
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df["content"])

# 3️⃣ Perfil del usuario (promedio)
user_indices = perfil_usuario.index
user_vector = tfidf_matrix[user_indices].mean(axis=0).A1  # <- clave

# 4️⃣ Similaridad (forma correcta)
similarity = cosine_similarity(tfidf_matrix, user_vector.reshape(1, -1))

df["similarity"] = similarity

# 5️⃣ Filtrar juegos ya poseídos
recomendaciones = df[~df["app_id"].isin(juegos_usuario)]

# 6️⃣ Mostrar resultados
print("\n🎮 Recomendaciones:")
print(recomendaciones.sort_values("similarity", ascending=False)[["name", "similarity"]].head(10))


🎮 Recomendaciones:
                                    name  similarity
74294                                活下去    0.507759
14440                              异世界物语    0.505600
14568                              百将群英传    0.505600
61231                              女神守护者    0.504440
62418  阿达三国志2019  竖版 Three Kingdoms 2019    0.504440
40080                               江湖问情    0.504440
83561                                时之扉    0.504200
54866                              三国梦之队    0.502750
12979                               幻世九州    0.500738
55049                               天书江湖    0.499216


In [36]:
# Juegos que el usuario ya posee

# Filtrar recomendaciones
recomendaciones = recomendaciones[~recomendaciones["app_id"].isin(juegos_usuario)]

# Mostrar top 10
print("\n🎮 Recomendaciones:")
print(recomendaciones[["name", "similarity"]].head(10))


🎮 Recomendaciones:
                                    name  similarity
0             Black Dragon Mage Playtest    0.000000
1  Supipara - Chapter 1 Spring Has Come!    0.025625
2      Mystery Solitaire The Black Raven    0.083920
3            버튜버 파라노이아 - Vtuber Paranoia    0.104079
4                          Maze Quest VR    0.066162
5                               Agony VR    0.107532
6                     Armored Brigade II    0.137974
7         Galacatraz: Eject Equip Escape    0.065311
8                            Hepta Beats    0.020597
9                 MUMBA IV: Egypt Jewels    0.061094
